In [33]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
import numpy as np
import random
from category_encoders import TargetEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [34]:
df = pd.read_csv("Mumbai House Prices.csv")

def convert(value,unit):
    if unit == "Cr":
        return value * 100
    elif unit == "L":
        return value
    else:
        return np.nan
        
df["price_lakh"] = df.apply(lambda x: convert(x['price'],x['price_unit']),axis = 1)
df=df.drop(columns=['price','price_unit','locality'])


In [35]:
encoder = OneHotEncoder(sparse_output=False)
encoded = encoder.fit_transform(df[['type','status','age']])

encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['type','status','age']))
df = pd.concat([df, encoded_df], axis=1).drop(['type','status','age'], axis=1)


In [36]:
x = df.drop('price_lakh', axis=1)
y = df['price_lakh']

In [37]:
def tr_te_split(X,y,test_size = 0.2,random_state = 42):
    random.seed(random_state)
    n = len(X)
    test_count = int(n*test_size)
    indices = list(range(n))
    random.shuffle(indices)
    train_indices = indices[:test_count]
    test_indices = indices[test_count:]

    X_train = X.iloc[train_indices].reset_index(drop=True)
    X_test  = X.iloc[test_indices].reset_index(drop=True)
    y_train = y.iloc[train_indices].reset_index(drop=True)
    y_test  = y.iloc[test_indices].reset_index(drop=True)

    return X_train,X_test,y_train,y_test

    


In [38]:
X_train, X_test, y_train, y_test = tr_te_split(x,y,test_size = 0.2,random_state = 42)

In [39]:
X_train,X_test,y_train,y_test

(       bhk  area          region  type_Apartment  type_Independent House  \
 0        1   690  Mira Road East             1.0                     0.0   
 1        1   720  Mira Road East             1.0                     0.0   
 2        2   620       Dronagiri             1.0                     0.0   
 3        3   996  Kandivali West             1.0                     0.0   
 4        1   514        Bhiwandi             1.0                     0.0   
 ...    ...   ...             ...             ...                     ...   
 15202    1   670      Thane West             1.0                     0.0   
 15203    2   961         Chembur             1.0                     0.0   
 15204    1   625  Mira Road East             1.0                     0.0   
 15205    2   590        Vikhroli             1.0                     0.0   
 15206    1   580     Nala Sopara             1.0                     0.0   
 
        type_Penthouse  type_Studio Apartment  type_Villa  \
 0           

In [40]:
encoder = TargetEncoder(cols=['region'])
X_train_encoded = encoder.fit_transform(X_train, y_train)
X_test_encoded = encoder.transform(X_test)

In [41]:
X_test_encoded

,bhk,area,region,type_Apartment,type_Independent House,type_Penthouse,type_Studio Apartment,type_Villa,status_Ready to move,status_Under Construction,age_New,age_Resale,age_Unknown
0,2,963,199.857530,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
1,1,610,42.073378,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,2,1045,59.263866,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
3,2,856,81.065717,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
4,1,650,81.065717,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
60826,2,884,205.013357,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
60827,3,1050,179.098970,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
60828,2,750,171.226239,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
60829,2,980,30.143332,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0


In [42]:
mean_area = X_test_encoded['area'].mean()
std_area = X_test_encoded['area'].std()
X_train_encoded['area'] =  ( X_train_encoded['area'] - mean_area )/std_area
X_test_encoded['area'] =  ( X_test_encoded['area'] - mean_area )/std_area

mean_region = X_test_encoded['region'].mean()
std_region = X_test_encoded['region'].std()
X_train_encoded['region'] =  ( X_train_encoded['region'] - mean_region )/std_region
X_test_encoded['region'] =  ( X_test_encoded['region'] - mean_region )/std_region


In [43]:
X_train_encoded.describe()



,bhk,area,region,type_Apartment,type_Independent House,type_Penthouse,type_Studio Apartment,type_Villa,status_Ready to move,status_Under Construction,age_New,age_Resale,age_Unknown
count,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000,15207.000000
mean,2.015519,0.003413,-0.000085,0.984021,0.000986,0.000066,0.012231,0.002696,0.585980,0.414020,0.506412,0.300125,0.193464
std,0.929033,1.010640,0.993643,0.125400,0.031392,0.008109,0.109920,0.051856,0.492568,0.492568,0.499975,0.458327,0.395026
min,1.000000,-1.341227,-1.219100,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,-0.574239,-0.721921,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.000000,-0.237841,0.049853,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,1.000000,0.000000,0.000000
75%,3.000000,0.225640,0.294937,1.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000
max,10.000000,22.390529,5.947855,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [44]:
X_train_encoded_df,X_test_encoded_df=X_train_encoded,X_test_encoded
y_train_copy,y_test_copy=y_train,y_test

In [45]:
X_train_encoded = X_train_encoded.values.astype(float)   # convert to numeric array
X_test_encoded = X_test_encoded.values.astype(float)   # convert to numeric array
y_train= y_train.values.reshape(-1, 1).astype(float)
y_test= y_test.values.reshape(-1, 1).astype(float)

# Add bias column
X_train_encoded = np.c_[np.ones(X_train_encoded.shape[0]), X_train_encoded]

# Initialize theta
theta = np.zeros((X_train_encoded.shape[1], 1))

#print(x.head(5))
print(y_train.shape)
print(theta.shape)
print(X_train_encoded.shape)


(15207, 1)
(14, 1)
(15207, 14)


In [52]:
def predict(x, theta):
    return np.dot(x, theta)

def cost_predict(x, y, theta):
    m = len(y)
    prediction = predict(x, theta)
    error = prediction - y
    cost = (1 / (2 * m)) * np.dot(error.T, error) 
    return cost

def gradient_descent(x, y, theta, alpha, iteration):
    m = len(y)
    cost_history = []
    for i in range(iteration):
        prediction = predict(x, theta)
        error = prediction - y
        theta -= (alpha/m) * np.dot(x.T, error) 
        cost = cost_predict(x, y, theta)
        cost_history.append(cost)

        if i % 100 == 0:
            print(f"Iteration {i}: Cost = {cost}")

    return theta, cost_history


alpha = 0.0001      
iterations = 1000  

theta, cost_history = gradient_descent(X_train_encoded, y_train, theta, alpha, iterations)
print("Final parameters:", theta)
print("Final cost", cost_history[-1])

y_pred = predict(X_train_encoded, theta)



Iteration 0: Cost = [[19044.61909928]]
Iteration 100: Cost = [[18184.45998196]]
Iteration 200: Cost = [[17422.65194718]]
Iteration 300: Cost = [[16746.06752148]]
Iteration 400: Cost = [[16143.39653886]]
Iteration 500: Cost = [[15604.89255076]]
Iteration 600: Cost = [[15122.1546829]]
Iteration 700: Cost = [[14687.93998248]]
Iteration 800: Cost = [[14296.0019921]]
Iteration 900: Cost = [[13940.95188341]]
Final parameters: [[1.58040404e+01]
 [4.95595185e+01]
 [2.53141102e+01]
 [2.12679654e+01]
 [1.54147868e+01]
 [5.02532742e-02]
 [1.68208529e-03]
 [1.70412436e-02]
 [3.20276932e-01]
 [1.02124246e+01]
 [5.59161575e+00]
 [6.90878176e+00]
 [4.58421660e+00]
 [4.31104201e+00]]
Final cost [[13621.22121543]]


In [53]:
X_test_encoded = np.c_[np.ones(X_test_encoded.shape[0]), X_test_encoded]
print(y_test.shape)
print(theta.shape)
print(X_test_encoded.shape)

(60831, 1)
(14, 1)
(60831, 14)


In [54]:
y_pred_test = predict(X_test_encoded, theta)


In [55]:
mae = mean_absolute_error(y_test, y_pred_test)
mse = mean_squared_error(y_test, y_pred_test)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred_test)

print(f"MAE: {mae:.3f}")
print(f"MSE: {mse:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R² Score: {r2:.3f}")

MAE: 64.305
MSE: 27529.297
RMSE: 165.920
R² Score: 0.419
